# Week 1 Práctica - Multi-Prompts

## Tutor que responde a preguntas técnicas sobre Python usando dos modelos de IA.

**Modelos implementados:**
- Modelo frontier en la nube (gpt-4o-mini)
- Modelo local (Llama 3.2 via Ollama)

 *La respuesta de GPT se muestra en streaming, palabra a palabra.*

## 1. Imports
Librerías necesarias para el funcionamiento del notebook.
Incluye el cliente de OpenAI, Ollama, y utilidades para mostrar Markdown en Jupyter.

In [1]:
# imports

from dotenv import load_dotenv                                     # Carga variables de entorno desde el fichero .env
from IPython.display import Markdown, display, update_display      # Muestra contenido Markdown en Jupyter
from openai import OpenAI                                          # Cliente oficial de OpenAI para llamadas a la API
import ollama                                                      # Cliente oficial de Ollama para modelos locales
import ipywidgets as widgets                                       # Entrada de datos
import os                                                          # Librería estándar de Python para interactuar con el SO

## 2. Constantes
Define los nombres de los modelos que vamos a usar.
Centralizar estos valores facilita cambiarlos en un único punto si es necesario.

In [2]:
# Constantes

MODEL_GPT  = 'gpt-4o-mini'   # Modelo OpenAI en la nube
MODEL_LLAMA = 'llama3.2'     # Modelo local via Ollama

# Constantes — control de respuesta
MAX_TOKENS  = 1200            # Límite máximo de tokens en la respuesta. Riesgo: Respuesta incompleta. Corta respuesta al alcanzar el número de tokens.
MAX_WORDS   = 300             # Límite orientativo de palabras indicado en el system prompt

## 3. Selección del modelo de IA

In [4]:
# Botones por modelo

boton_gpt = widgets.Button(
    description='A — GPT-4o-mini (OpenAI)',
    button_style='primary',                     # Color del botón
    layout=widgets.Layout(width='250px')
)

boton_llama = widgets.Button(
    description='B — Llama 3.2 (Ollama)',
    button_style='success',
    layout=widgets.Layout(width='250px')
)

# Crea el área de salida dentro del notebook donde mostrará los botones 
salida = widgets.Output()                      

 # Todo lo que esta dentro de este bloque aparece en el área de salida
with salida:                                    
    print("Selecciona el modelo de IA: 'A' o 'B'")
    print("")

def elegir_gpt(b):
    global MODEL
    MODEL = MODEL_GPT
    with salida:
        salida.clear_output()
        print(f"✓ Modelo seleccionado: {MODEL}")

def elegir_llama(b):
    global MODEL
    MODEL = MODEL_LLAMA
    with salida:
        salida.clear_output()
        print(f"✓ Modelo seleccionado: {MODEL}")

boton_gpt.on_click(elegir_gpt)
boton_llama.on_click(elegir_llama)

display(widgets.HBox([boton_gpt, boton_llama]), salida)

Output()

## 4. Configuración del entorno
Carga las variables del fichero .env, incluyendo la clave de API de OpenAI.
Inicializa el cliente OpenAI listo para hacer llamadas.

In [5]:
# Configurar entorno

load_dotenv()                           # Carga las variables del fichero .env del directorio raiz desde donde se inició Jupyter Lab
api_key = os.getenv('OPENAI_API_KEY')   # Obtiene la clave de API de OpenAI

# Inicializa el cliente según el modelo seleccionado
if MODEL == MODEL_GPT:
    if not api_key:
        print("⚠️  No se encontró la clave de API de OpenAI en el fichero .env")
    elif not api_key[:8]=='sk-proj-':
        print("⚠️  La clave no tiene un formato válido — debe empezar por sk-")
    elif api_key.strip() != api_key:
        print("⚠️  La clave tiene espacios al principio o al final — corrígelo en el .env")
    else:
        openai = OpenAI()
        print("✓ Cliente OpenAI inicializado correctamente")
else:
    print("✓ Usando Ollama local — no requiere API key")

✓ Usando Ollama local — no requiere API key


## 5. La pregunta
Define la pregunta técnica que el tutor va a responder.
Puedes modificar esta celda para preguntar sobre cualquier concepto Python.

In [6]:
# Registrar la pregunta

# Límite máximo de caracteres para la pregunta
MAX_CHARS = 1000

# Título
titulo = widgets.HTML("<h3>🎓 Tutor técnico de Python</h3><p>Escribe tu pregunta (máx. 1000 caracteres). Para salir escribe <b>SALIR</b>.</p>")

# Área de texto
texto = widgets.Textarea(
    placeholder='Escribe aquí tu pregunta...',
    layout=widgets.Layout(width='600px', height='120px')
)

# Contador de caracteres
contador = widgets.HTML("<small>0 / 1000 caracteres</small>")

# Botones
boton_enviar = widgets.Button(
    description='Enviar al tutor',
    layout=widgets.Layout(width='160px')
)
boton_enviar.style.button_color = '#2E7D32'

boton_borrar = widgets.Button(
    description='Borrar',
    layout=widgets.Layout(width='160px')
)
boton_borrar.style.button_color = '#B71C1C'

# Área de mensajes
salida = widgets.Output()

# Actualizar contador al escribir
def actualizar_contador(change):
    n = len(change['new'])
    color = 'red' if n > MAX_CHARS else 'gray'
    contador.value = f"<small style='color:{color}'>{n} / {MAX_CHARS} caracteres</small>"

texto.observe(actualizar_contador, names='value')

# Acción botón Enviar
def enviar(b):
    global question
    contenido = texto.value.strip()
    with salida:
        salida.clear_output()
        if contenido.upper() == 'SALIR':
            print("👋 Saliendo del tutor. ¡Hasta pronto!")
            boton_enviar.disabled = True
            boton_borrar.disabled = True
        elif not contenido:
            print("⚠️  Estamos esperando tu pregunta. Por favor escribe algo.")
        elif len(contenido) > MAX_CHARS:
            print(f"⚠️  La pregunta supera los {MAX_CHARS} caracteres. Por favor acórtala.")
        else:
            question = contenido
            print(f"✓ Pregunta registrada. Puedes ejecutar la siguiente celda.")

# Acción botón Borrar
def borrar(b):
    texto.value = ''
    with salida:
        salida.clear_output()
        print("🗑️  Pregunta borrada. Escribe una nueva.")

boton_enviar.on_click(enviar)
boton_borrar.on_click(borrar)

# Mostrar todo
display(
    titulo,
    texto,
    contador,
    widgets.HBox([boton_enviar, boton_borrar]),
    salida
)

HTML(value='<h3>🎓 Tutor técnico de Python</h3><p>Escribe tu pregunta (máx. 1000 caracteres). Para salir escrib…

Textarea(value='', layout=Layout(height='120px', width='600px'), placeholder='Escribe aquí tu pregunta...')

HTML(value='<small>0 / 1000 caracteres</small>')

Output()

## 6. System Prompt
Define el rol y comportamiento del modelo de IA.
Indica al modelo que actúe como tutor técnico experto en Python.

In [7]:
print(question)

Cómo se programa una ecuación matemática en Python?


In [8]:
# El prompt de sistema

system_prompt = """
Eres un tutor técnico experto en Python, ingeniería de software y LLMs.
Responde de forma detallada y didáctica, usando ejemplos prácticos de código.
Responde siempre en español y en formato Markdown.
Sé conciso — limita tu respuesta a un máximo de {MAX_WORDS} palabras.
"""

## 7. User Prompt
Construye el mensaje con la pregunta que se envía al modelo.
Combina un texto introductorio fijo con la pregunta definida en el paso anterior.

In [9]:
# El prompt de usuario

user_prompt = "Por favor, da una explicación detallada de la siguiente pregunta: " + question


## 8. Mensajes
Empaqueta los prompts en el formato estándar que esperan todos los LLMs.
Lista con dos elementos: rol system y rol user.

In [10]:
# Los mensajes en formato lista

messages = [
    {"role": "system", "content": system_prompt},

    #Ejemplo - pregunta sobre concepto básico
    {"role": "user", "content": "¿Qué es una variable en Python?"},
    {"role": "assistant", "content": """
## Variable en Python
Una variable es un contenedor que almacena un valor en memoria.

**Ejemplo:**
```python
nombre = "Chema"   # string
edad = 50          # entero
precio = 9.99      # decimal
```
A diferencia de Java, en Python no se declara el tipo — se infiere automáticamente.
    """},  

    # Ejemplo — pregunta sobre código concreto
    {"role": "user", "content": "¿Qué hace el operador ** en Python?"},
    {"role": "assistant", "content": """
## Operador ** en Python
El operador `**` calcula la potencia de un número.

**Ejemplo:**
```python
resultado = 2 ** 8   # 2 elevado a 8 = 256
raiz = 9 ** 0.5      # raíz cuadrada de 9 = 3.0
```
Es equivalente a `Math.pow(2, 8)` en Java.
    """},
    
    {"role": "user",   "content": user_prompt}
]

## 9. Respuesta GPT-4o-mini con streaming
Llama a la API de OpenAI y muestra la respuesta progresivamente en Markdown.
El streaming mejora la experiencia — el usuario ve la respuesta mientras se genera.

In [11]:
# Llamada a gpt-4o-mini con streaming

if MODEL == MODEL_GPT:
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        stream=True,        # activa el modo streaming
        max_tokens=MAX_TOKENS
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```", "").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)



## 10. Respuesta Llama 3.2 (Ollama local)
Llama al modelo local usando la librería ollama.
La respuesta se recibe completa y se muestra formateada en Markdown.

In [12]:
# Llamada a llama 3.2 via Ollama 

if MODEL == MODEL_LLAMA:
    response = ollama.chat(
        model=MODEL_LLAMA,
        messages=messages
    )
    reply = response['message']['content']
    display(Markdown(reply))

## Ecuaciones Matemáticas en Python
Python ofrece varias formas de programar y resolver ecuaciones matemáticas. Aquí te presentamos algunas opciones:

### 1. Utilizando `sympy`

`Sympy` es una biblioteca Python para la manipulación simbólica de expresiones matemáticas. Puedes utilizarla para definir variables, crear ecuaciones y resolver problemas.

**Ejemplo:**
```python
from sympy import symbols, Eq, solve

# Definimos las variables
x = symbols('x')

# Creamos la ecuación
ecuacion = Eq(x + 3, 7)

# Resolvemos el problema
solution = solve(ecuacion, x)

print(solution)  # Imprime: [4]
```
En este ejemplo, definimos una variable `x` y creamos una ecuación `x + 3 = 7`. Luego, utilizamos la función `solve()` para resolver el problema y obtener la solución.

### 2. Utilizando `numexpr`

`Numexpr` es una biblioteca Python que se utiliza para evaluar expresiones matemáticas numéricas. Puedes utilizarla para realizar cálculos rápidos y precisos.

**Ejemplo:**
```python
import numexpr as ne

# Definimos la ecuación
ecuacion = "x + 3"

# Evaluamos la ecuación
resultado = ne.parse(equacion).eval({"x": 4})

print(resultado)  # Imprime: 7
```
En este ejemplo, definimos una variable `x` y creamos una ecuación `x + 3`. Luego, utilizamos la función `parse()` para evaluar la ecuación y obtener el resultado.

### 3. Utilizando `eval()`

La función `eval()` es una función de Python que evalúa una expresión matemática. Puedes utilizarla para realizar cálculos simples.

**Ejemplo:**
```python
ecuacion = "x + 3"

# Evaluamos la ecuación
resultado = eval(equacion, {"x": 4})

print(resultado)  # Imprime: 7
```
En este ejemplo, definimos una variable `x` y creamos una ecuación `x + 3`. Luego, utilizamos la función `eval()` para evaluar la ecuación y obtener el resultado. Sin embargo, ten en cuenta que utilizar `eval()` puede ser peligroso si no se utiliza con precaución, ya que permite la ejecución de código arbitrario.

En resumen, existen varias formas de programar ecuaciones matemáticas en Python, dependiendo de las necesidades y del nivel de complejidad. Las opciones más populares son `sympy` y `numexpr`, aunque también puedes utilizar `eval()` para cálculos simples.